In [0]:
# python imports
from pyspark.sql import functions as F
import scipy.stats as stats
import numpy as np
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import plotly.express as px
import scipy.sparse as sp
%pip install openpyxl
import openpyxl 

In [0]:
import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────
PSD_PATH    = "/Volumes/formica_workspace/benchmark_validation/input_files/psd_alldata.csv"
MARKET_YEAR = 2025

PSD_TO_HS4 = {
    "Cotton":             "5201",
    "Wheat":              "1001",
    "Oil, Palm":          "1511",
    "Oilseed, Soybean":   "1201",
    "Meal, Soybean":      "2304",
    "Oil, Soybean":       "1507",
    "Sugar, Centrifugal": "1701",
}
ATTRIBUTES = ["Production", "Imports", "Exports"]

UNIT_TO_TONNES = {                       # PSD reports in thousands
    "(1000 MT)":            1000,
    "(1000 480 lb. Bales)": 1000 * 0.21772,
}

PSD_TO_ISO2 = {                          # PSD uses FIPS 10-4 codes, not ISO
    "AF": "AF", "AG": "DZ", "AJ": "AZ", "AL": "AL", "AM": "AM", "AO": "AO",
    "AR": "AR", "AS": "AU", "AU": "AT", "BA": "BH", "BE": "BE", "BG": "BD",
    "BK": "BA", "BM": "MM", "BO": "BY", "BR": "BR", "CA": "CA", "CB": "KH",
    "CD": "TD", "CH": "CN", "CI": "CL", "CM": "CM", "CS": "CR", "CT": "CF",
    "CY": "CY", "DM": "BJ", "DR": "DO", "EG": "EG", "EI": "IE", "EN": "EE",
    "ES": "SV", "ET": "ET", "EZ": "CZ", "FR": "FR", "GH": "GH", "GM": "DE",
    "GR": "GR", "GT": "GT", "GU": "GN", "HA": "HT", "HO": "HN", "HR": "HR",
    "HU": "HU", "ID": "ID", "IN": "IN", "IT": "IT", "IV": "CI", "JA": "JP",
    "KG": "KG", "KS": "KR", "KZ": "KZ", "LE": "LB", "LG": "LV", "LH": "LT",
    "LO": "SK", "LT": "LS", "MA": "MG", "MD": "MD", "MI": "MW", "ML": "ML",
    "MX": "MX", "MY": "MY", "MZ": "MZ", "NG": "NE", "NI": "NG", "NL": "NL",
    "NU": "NI", "PA": "PY", "PK": "PK", "PL": "PL", "PN": "PA", "PO": "PT",
    "RH": "ZW", "RO": "RO", "SG": "SN", "SI": "SI", "SU": "SD", "SY": "SY",
    "TI": "TJ", "TO": "TG", "TU": "TR", "TX": "TM", "TZ": "TZ", "UG": "UG",
    "US": "US", "UV": "BF", "UZ": "UZ", "VM": "VN", "YM": "YE", "ZA": "ZM",
    "BL": "BO", "BU": "BG", "BY": "BI", "CE": "LK", "CF": "CG", "CG": "CD",
    "CO": "CO", "DA": "DK", "EC": "EC", "FI": "FI", "HK": "HK", "IR": "IR",
    "IS": "IL", "IZ": "IQ", "JO": "JO", "KE": "KE", "KN": "KP", "KU": "KW",
    "MO": "MA", "MU": "OM", "NO": "NO", "NZ": "NZ", "PE": "PE", "RP": "PH",
    "RS": "RU", "SA": "SA", "SF": "ZA", "SN": "SG", "SP": "ES", "SW": "SE",
    "SZ": "CH", "TH": "TH", "TS": "TN", "TW": "TW", "UK": "GB", "UP": "UA",
    "UY": "UY", "VE": "VE",
}

# ── 1. Load and filter PSD ───────────────────────────────────────────────
psd = pd.read_csv(PSD_PATH)
psd = psd[
    psd["Attribute_Description"].isin(ATTRIBUTES)
    & (psd["Market_Year"] == MARKET_YEAR)
    & psd["Commodity_Description"].isin(PSD_TO_HS4)
]

unknown_units = psd.loc[~psd["Unit_Description"].isin(UNIT_TO_TONNES), "Unit_Description"].unique()
psd["value_t"] = psd["Value"] * psd["Unit_Description"].map(UNIT_TO_TONNES)

# ── 2. Pivot to one row per country-commodity ────────────────────────────
psd = (psd.pivot_table(index=["Country_Code", "Country_Name", "Commodity_Description"],
                       columns="Attribute_Description", values="value_t", aggfunc="first")
          .reset_index()
          .rename_axis(columns=None)
          .rename(columns={"Commodity_Description": "commodity",
                           "Production": "production",
                           "Imports": "imports",
                           "Exports": "exports"}))
psd["hs_code"] = psd["commodity"].map(PSD_TO_HS4)

# ── 3. Global production share (before dropping unmapped countries) ──────
psd["global_production"]       = psd.groupby("hs_code")["production"].transform("sum")
psd["global_production_share"] = psd["production"] / psd["global_production"]

# ── 4. Domestic share metrics ────────────────────────────────────────────
passthrough = np.minimum(psd["exports"], psd["imports"])

psd["domestic_share"]     = psd["production"] / (psd["production"] + psd["imports"])

# ── 5. Map to ISO and filter ─────────────────────────────────────────────
psd["country_code"] = psd["Country_Code"].map(PSD_TO_ISO2)

unmapped = psd.loc[psd["country_code"].isna(), ["Country_Code", "Country_Name"]].drop_duplicates()

benchmarks = (psd[psd["country_code"].notna() & psd["domestic_share"].notna()]
                [["hs_code", "commodity", "country_code",
                  "domestic_share", "global_production_share"]]
                .sort_values("global_production_share", ascending=False))
benchmarks

In [0]:
import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────
USGS_PATH   = "/Volumes/formica_workspace/benchmark_validation/input_files/2024MYBv3_Production_table (1).csv"
TRADE_TABLE = "silver_country_hs_code_trade_weight_summary"

KG_TO_T = 0.001
TOL     = 0.25                                          # exports allowed up to 125% of supply

USGS_TO_ISO2 = {
    "Afghanistan": "AF", "Albania": "AL", "Algeria": "DZ", "Argentina": "AR",
    "Armenia": "AM", "Australia": "AU", "Austria": "AT", "Azerbaijan": "AZ",
    "Bahrain": "BH", "Bolivia": "BO", "Bosnia and Herzegovina": "BA",
    "Botswana": "BW", "Brazil": "BR", "Bulgaria": "BG", "Burkina Faso": "BF",
    "Burma": "MM", "Burundi": "BI", "Cambodia": "KH", "Cameroon": "CM",
    "Canada": "CA", "Central African Republic": "CF", "Chile": "CL",
    "China": "CN", "Colombia": "CO", "Congo (Brazzaville)": "CG",
    "Congo (Kinshasa)": "CD", "Côte d'Ivoire": "CI", "Croatia": "HR",
    "Cuba": "CU", "Cyprus": "CY", "Czechia": "CZ", "Dominican Republic": "DO",
    "Ecuador": "EC", "Egypt": "EG", "Eritrea": "ER", "Eswatini": "SZ",
    "Ethiopia": "ET", "Fiji": "FJ", "Finland": "FI", "French Guiana": "GF",
    "Gabon": "GA", "Georgia": "GE", "Germany": "DE", "Ghana": "GH",
    "Greece": "GR", "Guinea": "GN", "Guyana": "GY", "India": "IN",
    "Indonesia": "ID", "Iran": "IR", "Ireland": "IE", "Israel": "IL",
    "Jamaica": "JM", "Japan": "JP", "Jordan": "JO", "Kazakhstan": "KZ",
    "Kenya": "KE", "Korea, Republic Of": "KR", "Kosovo": "XK",
    "Kyrgyzstan": "KG", "Laos": "LA", "Lebanon": "LB", "Liberia": "LR",
    "Madagascar": "MG", "Malawi": "MW", "Malaysia": "MY", "Mali": "ML",
    "Mauritania": "MR", "Mexico": "MX", "Mongolia": "MN", "Montenegro": "ME",
    "Morocco": "MA", "Mozambique": "MZ", "Namibia": "NA", "Nauru": "NR",
    "New Caledonia": "NC", "New Zealand": "NZ", "Niger": "NE", "Nigeria": "NG",
    "North Korea": "KP", "North Macedonia": "MK", "Norway": "NO", "Oman": "OM",
    "Pakistan": "PK", "Papua New Guinea": "PG", "Peru": "PE",
    "Philippines": "PH", "Poland": "PL", "Portugal": "PT", "Romania": "RO",
    "Russia": "RU", "Rwanda": "RW", "Saudi Arabia": "SA", "Senegal": "SN",
    "Serbia": "RS", "Sierra Leone": "SL", "Slovakia": "SK",
    "Solomon Islands": "SB", "South Africa": "ZA", "Spain": "ES",
    "Sri Lanka": "LK", "Sudan": "SD", "Suriname": "SR", "Sweden": "SE",
    "Syria": "SY", "Taiwan": "TW", "Tajikistan": "TJ", "Tanzania": "TZ",
    "Thailand": "TH", "Togo": "TG", "Tunisia": "TN", "Turkey": "TR",
    "Uganda": "UG", "Ukraine": "UA", "United Arab Emirates": "AE",
    "United Kingdom": "GB", "United States": "US", "Uzbekistan": "UZ",
    "Venezuela": "VE", "Vietnam": "VN", "Zambia": "ZM", "Zimbabwe": "ZW",
}

USGS_TO_HS4 = {
    "Iron ore":       "2601",
    "Bauxite":        "2606",
    "Chromium":       "2610",
    "Phosphate rock": "2510",
}
BASIS_EXCLUDE = ["imported", "laterite", "cement", "guano"]

BASIS_PRIORITY = {
    "Usable": 0, "Wet basis": 0,
    "": 1, "All grades": 1, "Chromite": 1,
    "Dry basis": 2, "Dry bauxite equivalent of crude": 2,
    "Crude": 3, "Run of mine": 3,
}
COMPONENT_BASES = [
    "Metallurgical grade", "Nonmetallurgical grade",          # Saudi bauxite
    "Chromite, less than 44% Cr", "Chromite, 44% to 48% Cr",  # South Africa chromite
]
FORM_PRIORITY = {"Chromite concentrates": 0, "": 1, "Chromite ore": 2}

# rows labelled as content whose values are clearly gross tonnage (see note above)
GROSS_OVERRIDES = [("South Africa", "Chromium")]


# ── 1. USGS production (gross weight) ────────────────────────────────────
usgs = pd.read_csv(USGS_PATH)

override = (usgs.set_index(["Country (Short Form) or Locality", "Level 2 (Commodity)"])
                .index.isin(GROSS_OVERRIDES))

usgs = usgs[
    ((usgs["Commodity Unit"] == "Gross weight") | override)
    & (usgs["Level 2 (Commodity)"].isin(USGS_TO_HS4))
    & (usgs["Level 3 (Type)"].fillna("Mine") == "Mine")
]
usgs = usgs.rename(columns={
    "Country (Short Form) or Locality": "country",
    "Level 2 (Commodity)":              "commodity",
    "Level 5 (Form)":                   "form",
    "Descriptor":                       "basis",
    "Time Period":                      "date",
    "Value":                            "value",
    "Unit":                             "unit",
})[["country", "commodity", "form", "basis", "date", "value", "unit"]]
usgs[["form", "basis"]] = usgs[["form", "basis"]].fillna("").apply(lambda s: s.str.strip())
usgs = usgs[usgs["value"] > 0]
usgs[["form", "basis"]] = usgs[["form", "basis"]].fillna("").apply(lambda s: s.str.strip())
usgs = usgs[usgs["value"] > 0]
usgs = usgs[~usgs["basis"].str.contains("|".join(BASIS_EXCLUDE), case=False)]

# ── 2. Latest year, one row per country-commodity ────────────────────────
key  = ["country", "commodity", "date"]
usgs = usgs[usgs["date"] == usgs.groupby(["country", "commodity"])["date"].transform("max")]

# sum component grades (Saudi bauxite, South African chromite)
is_comp = usgs["basis"].isin(COMPONENT_BASES)
comp = (usgs[is_comp]
          .groupby(key + ["form", "unit"], as_index=False)["value"].sum()
          .assign(basis="All grades"))
usgs = pd.concat([usgs[~is_comp], comp], ignore_index=True)

dupes = usgs[usgs.duplicated(key, keep=False)].sort_values(key)   # diagnostic: multiple forms/bases

# one row per key: preferred form first, then preferred basis
usgs = (usgs.assign(form_rank=usgs["form"].map(FORM_PRIORITY).fillna(9),
                    basis_rank=usgs["basis"].map(BASIS_PRIORITY).fillna(9))
            .sort_values(["form_rank", "basis_rank"])
            .drop_duplicates(key)
            .drop(columns=["form_rank", "basis_rank"]))

usgs["country_code"] = usgs["country"].map(USGS_TO_ISO2)
usgs["hs_code"]      = usgs["commodity"].map(USGS_TO_HS4)

unmapped = usgs.loc[usgs["country_code"].isna(), "country"].unique()
usgs["country_code"] = usgs["country"].map(USGS_TO_ISO2)
usgs["hs_code"]      = usgs["commodity"].map(USGS_TO_HS4)

# world total includes every country USGS reports, mapped or not, pass or fail
usgs["global_production"]       = usgs.groupby("commodity")["value"].transform("sum")
usgs["global_production_share"] = usgs["value"] / usgs["global_production"]

unmapped = usgs.loc[usgs["country_code"].isna(), "country"].unique()
usgs = usgs.dropna(subset=["country_code", "hs_code", "value"])
usgs = usgs.dropna(subset=["country_code", "hs_code", "value"])

# ── 3. Comtrade trade weights (kg → t) ───────────────────────────────────
hs_list = ", ".join(f"'{c}'" for c in USGS_TO_HS4.values())
trade = spark.sql(f"SELECT * FROM {TRADE_TABLE} WHERE hs_code IN ({hs_list})").toPandas()
trade["hs_code"] = trade["hs_code"].astype(str)
trade[["import_weight", "export_weight"]] *= KG_TO_T

# ── 4. Merge and compute metrics ─────────────────────────────────────────
bench  = usgs.merge(trade, how="left", on=["hs_code", "country_code"])
supply = bench["value"] + bench["import_weight"].fillna(0)

bench["domestic_share"] = bench["value"] / supply
bench["export_ratio"]   = bench["export_weight"] / supply
bench["mass_balance_status"] = np.select(
    [bench["export_weight"].isna(),
     bench["export_ratio"] <= 1,
     bench["export_ratio"] <= 1 + TOL],
    ["no_trade_data", "ok", "within_tolerance"],
    default="fail",
)
bench["mass_balances"] = bench["mass_balance_status"].isin(["ok", "within_tolerance"])

# ── 5. Validated benchmark rows ──────────────────────────────────────────
valid = (bench[bench["mass_balances"]]
           [["hs_code", "commodity", "country_code", "domestic_share", "global_production_share"]]
           .sort_values(["hs_code", "global_production_share"], ascending=[True, False]))
valid


In [0]:
final_benchmarks = pd.concat([benchmarks, valid])
final_benchmarks.sort_values('global_production_share', ascending = False)
final_benchmarks.commodity.unique()